# 🤖 03 — TF-IDF + Cosine Similarity (Baseline)
**Skripsi:** Sistem Rekomendasi Dosen Pembimbing Berbasis NLP — Teknik Informatika Unila

---

### Alur Notebook Ini
```
[profil_bersih dosen]  → TF-IDF Vectorizer → Matrix Dosen  ┐
                                                             ├→ Cosine Similarity → Top-K Rekomendasi
[teks_bersih skripsi] → TF-IDF Transform  → Vektor Query   ┘
```

Hasil akhir:
- Model TF-IDF tersimpan di `models/tfidf_model.pkl`
- Evaluasi Top-1 / Top-3 / Top-5 Accuracy
- Fungsi demo rekomendasi interaktif

> ⚠️ **Pastikan Notebook 02 sudah selesai sebelum menjalankan ini.**

---
## 🔧 LANGKAH 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
PROJECT_NAME = 'skripsi-rekomendasi-dosen'
ROOT = f'/content/drive/MyDrive/{PROJECT_NAME}'
sys.path.insert(0, os.path.join(ROOT, 'src'))

import config
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('✅ Semua library siap.')

---
## 📌 LANGKAH 1 — Load Data Bersih

In [ ]:
df_skripsi = pd.read_csv(config.FILE_SKRIPSI_CLEAN)
df_dosen   = pd.read_csv(config.FILE_DOSEN_CLEAN)

# Pastikan tidak ada NaN
df_skripsi['teks_bersih']   = df_skripsi['teks_bersih'].fillna('')
df_dosen['profil_bersih']   = df_dosen['profil_bersih'].fillna('')

print('✅ Data dimuat.')
print(f'   Skripsi : {len(df_skripsi)} baris')
print(f'   Dosen   : {len(df_dosen)} dosen')
print()
print('Daftar dosen yang terdaftar:')
for i, nama in enumerate(df_dosen['nama_dosen'].tolist(), 1):
    print(f'  {i:2}. {nama}')

---
## 📌 LANGKAH 2 — Bangun TF-IDF Vectorizer

In [ ]:
# ─── GABUNGKAN SEMUA TEKS untuk FITTING ──────────────────────────
# TF-IDF harus fit pada gabungan dokumen dosen + skripsi
# agar vocabulary mencakup semua kata yang relevan

corpus_fit = df_dosen['profil_bersih'].tolist() + df_skripsi['teks_bersih'].tolist()

print(f'Total dokumen untuk fitting TF-IDF: {len(corpus_fit)}')
print(f'  - Profil dosen  : {len(df_dosen)}')
print(f'  - Judul skripsi : {len(df_skripsi)}')

In [ ]:
# ─── FIT TF-IDF VECTORIZER ───────────────────────────────────────
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),    # unigram + bigram
    min_df=1,              # minimal muncul di 1 dokumen
    max_df=0.95,           # abaikan kata yang muncul di >95% dokumen
    sublinear_tf=True,     # gunakan log TF untuk mengurangi dominasi kata sangat sering
)

vectorizer.fit(corpus_fit)

print(f'✅ TF-IDF Vectorizer berhasil difit.')
print(f'   Ukuran vocabulary  : {len(vectorizer.vocabulary_):,} term')
print(f'   ngram_range        : (1, 2) — unigram & bigram')
print(f'   sublinear_tf       : True')
print()

# Contoh 10 term dari vocabulary
sample_vocab = list(vectorizer.vocabulary_.keys())[:20]
print(f'Contoh 20 term dari vocabulary: {sample_vocab}')

In [ ]:
# ─── TRANSFORM PROFIL DOSEN → MATRIX TF-IDF ─────────────────────
# Setiap baris = satu dosen, setiap kolom = satu term
tfidf_dosen = vectorizer.transform(df_dosen['profil_bersih'])

print(f'✅ Matrix TF-IDF Dosen: {tfidf_dosen.shape}')
print(f'   Baris (dosen) : {tfidf_dosen.shape[0]}')
print(f'   Kolom (term)  : {tfidf_dosen.shape[1]:,}')
print(f'   Sparsity      : {(1 - tfidf_dosen.nnz / (tfidf_dosen.shape[0]*tfidf_dosen.shape[1]))*100:.1f}%')

In [ ]:
# ─── SIMPAN MODEL TF-IDF ─────────────────────────────────────────
model_path = config.FILE_TFIDF_MODEL
with open(model_path, 'wb') as f:
    pickle.dump({'vectorizer': vectorizer, 'tfidf_dosen': tfidf_dosen,
                 'nama_dosen': df_dosen['nama_dosen'].tolist()}, f)

print(f'💾 Model TF-IDF tersimpan: {model_path}')

---
## 📌 LANGKAH 3 — Fungsi Rekomendasi

In [ ]:
def rekomendasikan_tfidf(judul_query, top_k=5, verbose=True):
    """
    Rekomendasikan dosen pembimbing berdasarkan judul skripsi.

    Parameters:
        judul_query : str  — judul atau topik skripsi mahasiswa
        top_k       : int  — jumlah rekomendasi yang dikembalikan
        verbose     : bool — tampilkan output terformat

    Returns:
        list of (nama_dosen, score)
    """
    # Transform query menggunakan vectorizer yang sama
    query_vec = vectorizer.transform([judul_query])

    # Hitung cosine similarity antara query dan semua profil dosen
    scores = cosine_similarity(query_vec, tfidf_dosen).flatten()

    # Urutkan dari skor tertinggi
    ranking = np.argsort(scores)[::-1]

    hasil = [(df_dosen['nama_dosen'].iloc[i], round(float(scores[i]), 4)) for i in ranking[:top_k]]

    if verbose:
        print(f'🔍 Query: "{judul_query}"')
        print(f'📋 Top-{top_k} Rekomendasi Dosen Pembimbing (TF-IDF):')
        print('-' * 65)
        for rank, (nama, score) in enumerate(hasil, 1):
            bar = '█' * int(score * 30)
            print(f'  {rank}. {nama[:40]:<42} Skor: {score:.4f} {bar}')
        print()

    return hasil

print('✅ Fungsi rekomendasikan_tfidf() siap.')

In [ ]:
# ─── UJI COBA REKOMENDASI ─────────────────────────────────────────
contoh_judul = [
    'Implementasi Deep Learning untuk Deteksi Penyakit Tanaman pada Citra Digital',
    'Pembangunan Sistem Informasi Manajemen Arsip Berbasis Web',
    'Klasifikasi Teks Bahasa Indonesia Menggunakan BERT',
]

for judul in contoh_judul:
    rekomendasikan_tfidf(judul, top_k=5)
    print()

---
## 📌 LANGKAH 4 — Evaluasi Top-K Accuracy

In [ ]:
# ─── SPLIT DATA: TRAIN / TEST ────────────────────────────────────
# Gunakan data skripsi sebagai ground truth:
#   - Query    : judul skripsi (teks_bersih)
#   - GT label : nama dosen pembimbing asli
#
# Strategi split: 80% untuk fitting profil dosen sudah dilakukan.
# Di sini kita evaluasi menggunakan SEMUA data skripsi sebagai test set
# (karena profil dosen dibangun dari publikasi, bukan dari data skripsi itu sendiri)

from sklearn.model_selection import train_test_split

# Filter hanya skripsi yang pembimbingnya ada di daftar dosen
dosen_valid = set(df_dosen['nama_dosen'].tolist())
df_eval = df_skripsi[df_skripsi['pembimbing'].isin(dosen_valid)].copy().reset_index(drop=True)

print(f'Total data skripsi              : {len(df_skripsi)}')
print(f'Data dengan dosen terdaftar     : {len(df_eval)}')
print(f'Data tidak bisa dievaluasi      : {len(df_skripsi) - len(df_eval)}')
print()

# Stratified split 80:20 berdasarkan dosen
df_train, df_test = train_test_split(
    df_eval, test_size=0.2, random_state=42, stratify=df_eval['pembimbing']
)
print(f'Data train : {len(df_train)} skripsi')
print(f'Data test  : {len(df_test)} skripsi  ← digunakan untuk evaluasi')

In [ ]:
# ─── FUNGSI EVALUASI TOP-K ACCURACY ─────────────────────────────
def evaluasi_topk(df_test, metode='tfidf', top_k_list=[1, 3, 5]):
    """
    Hitung Top-K Accuracy untuk setiap nilai K.

    Top-K Accuracy = jumlah prediksi benar dalam K teratas / total data test
    'Benar' = nama dosen pembimbing asli ada di dalam Top-K rekomendasi.
    """
    results = {k: 0 for k in top_k_list}
    max_k   = max(top_k_list)
    errors  = []  # simpan kasus yang salah untuk analisis

    for _, row in df_test.iterrows():
        query   = row['teks_bersih']
        gt      = row['pembimbing']  # ground truth

        # Ambil rekomendasi top max_k
        reko    = rekomendasikan_tfidf(query, top_k=max_k, verbose=False)
        nama_reko = [r[0] for r in reko]

        for k in top_k_list:
            if gt in nama_reko[:k]:
                results[k] += 1
            elif k == 1:
                errors.append({'judul': row['judul'], 'gt': gt, 'pred': nama_reko[0]})

    n = len(df_test)
    accuracy = {k: round(v / n * 100, 2) for k, v in results.items()}
    return accuracy, errors

print('✅ Fungsi evaluasi siap.')

In [ ]:
# ─── JALANKAN EVALUASI ────────────────────────────────────────────
print('⏳ Mengevaluasi model TF-IDF pada data test...\n')
acc_tfidf, errors = evaluasi_topk(df_test, metode='tfidf', top_k_list=[1, 3, 5])

print('=' * 45)
print('📊 HASIL EVALUASI — TF-IDF + Cosine Similarity')
print('=' * 45)
print(f'  Data test  : {len(df_test)} skripsi')
print()
for k, acc in acc_tfidf.items():
    bar = '█' * int(acc / 3)
    print(f'  Top-{k} Accuracy : {acc:6.2f}%  {bar}')
print('=' * 45)

In [ ]:
# ─── ANALISIS ERROR (TOP-1) ───────────────────────────────────────
print(f'\n🔎 Analisis Kesalahan Top-1 ({len(errors)} kasus):\n')
df_err = pd.DataFrame(errors)
if not df_err.empty:
    print(df_err[['judul','gt','pred']].head(10).to_string(index=False))
    print()
    # Dosen mana yang paling sering salah diprediksi
    print('Dosen paling sering missed (GT tidak terpilih di Top-1):')
    print(df_err['gt'].value_counts().to_string())

In [ ]:
# ─── VISUALISASI HASIL ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grafik 1: Bar chart Top-K Accuracy
ks     = [f'Top-{k}' for k in acc_tfidf.keys()]
accs   = list(acc_tfidf.values())
bars   = axes[0].bar(ks, accs, color=['#1976D2','#388E3C','#F57C00'], width=0.4, edgecolor='white')
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=12)
axes[0].set_ylim(0, 110)
axes[0].set_title('Top-K Accuracy — TF-IDF Baseline', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_xlabel('Nilai K')
axes[0].axhline(y=100, color='gray', linestyle='--', alpha=0.4)

# Grafik 2: Heatmap Cosine Similarity antar dosen (visualisasi kedekatan profil)
sim_matrix = cosine_similarity(tfidf_dosen).round(3)
nama_pendek = [n.split(',')[0].split('.')[-1].strip()[:15] for n in df_dosen['nama_dosen']]
sns.heatmap(sim_matrix, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=nama_pendek, yticklabels=nama_pendek,
            ax=axes[1], cbar_kws={'label': 'Cosine Similarity'},
            annot_kws={'size': 7})
axes[1].set_title('Heatmap Kemiripan Profil Antar Dosen', fontweight='bold', fontsize=13)
axes[1].tick_params(axis='x', rotation=45)
axes[1].tick_params(axis='y', rotation=0)

plt.tight_layout()
plot_path = os.path.join(config.RESULTS_DIR, 'tfidf_evaluation.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'\n📊 Grafik tersimpan: {plot_path}')

In [ ]:
# ─── SIMPAN HASIL EVALUASI ────────────────────────────────────────
df_hasil = pd.DataFrame([{
    'metode'          : 'TF-IDF + Cosine Similarity',
    'top1_accuracy'   : acc_tfidf[1],
    'top3_accuracy'   : acc_tfidf[3],
    'top5_accuracy'   : acc_tfidf[5],
    'n_test'          : len(df_test),
}])

df_hasil.to_csv(config.FILE_EVALUATION, index=False)
print(f'💾 Hasil evaluasi tersimpan: {config.FILE_EVALUATION}')
df_hasil

---
## 📌 LANGKAH 5 — Demo Interaktif

In [ ]:
# ─── DEMO: MASUKKAN JUDUL SKRIPSI SENDIRI ────────────────────────
# Ganti teks di bawah dengan judul skripsi yang ingin kamu cek

JUDUL_QUERY = "Sistem Deteksi Hoaks Berbasis Natural Language Processing Menggunakan IndoBERT"

print('=' * 65)
rekomendasikan_tfidf(JUDUL_QUERY, top_k=5)
print('=' * 65)

---
## ✅ Selesai — Ringkasan Notebook 03

| Output | Lokasi |
|--------|--------|
| Model TF-IDF | `models/tfidf_model.pkl` |
| Hasil evaluasi | `results/evaluation.csv` |
| Grafik evaluasi | `results/tfidf_evaluation.png` |

### 🗺️ Langkah Berikutnya:
> **`04_bert_embedding.ipynb`** — IndoBERT / Sentence-BERT embedding + evaluasi